--------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS:
import os, re, subprocess, time
import numpy as np
import matplotlib.pyplot as plt
import warnings
from nilearn.datasets import load_mni152_template
import nilearn.plotting as plotting
import mne
import nibabel as nib
from nibabel.processing import resample_from_to
import csv
import pandas as pd
from typing import Tuple
import glob

import json
import shutil

from nilearn import datasets
from nilearn.image import resample_to_img
from nilearn.maskers import NiftiMasker
import nilearn.plotting as plotting

In [ ]:
##### SET UP ENVIRONMENTAL VARIABLES FOR FREESURFER:
freesurfer_config = config.get("freesurfer", {})
FREESURFER_HOME = Path(freesurfer_config.get("home", "/opt/freesurfer-7.4.1")).expanduser()
FS_LICENSE = Path(freesurfer_config.get("license", FREESURFER_HOME / "license.txt")).expanduser()
SUBJECTS_DIR = Path(freesurfer_config.get("subjects_dir", FREESURFER_HOME / "subjects")).expanduser()
CHECK_FS_VERSION = bool(freesurfer_config.get("check_version", True))
if not FREESURFER_HOME.exists():
    raise FileNotFoundError(f"FREESURFER_HOME not found: {FREESURFER_HOME}")
if not FS_LICENSE.exists():
    raise FileNotFoundError(f"FreeSurfer license file not found: {FS_LICENSE}")
if not SUBJECTS_DIR.exists():
    raise FileNotFoundError(f"FreeSurfer SUBJECTS_DIR not found: {SUBJECTS_DIR}")
os.environ["FREESURFER_HOME"] = str(FREESURFER_HOME)
os.environ["FS_LICENSE"] = str(FS_LICENSE)
os.environ["SUBJECTS_DIR"] = str(SUBJECTS_DIR)
fs_bin_dir = FREESURFER_HOME / "bin"
os.environ["PATH"] = f"{fs_bin_dir}:{os.environ.get('PATH', '')}"

subprocess.run("recon-all --version", shell=True, check=True)

In [ ]:
# --------------------------------------------------------------------
### SET PARAMETERS:

# -------------------------
# General parameters
# -------------------------
HARD_STOP   = bool(config["hard_errors"])
RANDOM_SEED = int(config["random_seed"])
SUBSET      = config.get("subset", None)

# Optional: silence MNE filename warnings
warnings.filterwarnings(
    "ignore",
    message="This filename .* does not conform to MNE naming conventions.*",
    category=RuntimeWarning)

OVERWRITE_CACHED_ATLAS  = bool(config.get("overwrite_cached_atlas",  True))
OVERWRITE_PARCELLATIONS = bool(config.get("overwrite_parcellations", True))

# -------------------------
# Processing / parcellation parameters
# -------------------------
ENABLE_QC_OVERLAYS = bool(config.get("enable_QC_overlays", False))

PARCELLATION = config.get("parcellation", {})
ATLAS_FAMILY = str(PARCELLATION.get("atlas", "")).strip()
AF_LOWER     = ATLAS_FAMILY.lower()

ATLAS_N_ROIS = int(PARCELLATION.get("n_rois", 0))
ATLAS_NETWORK_SCALE = PARCELLATION.get("network_scale", None)

# Set ANTS registration type:
REG_TYPE = config['registration_parameters']['registration_type']

# Time-series type validation (must be 'mean' or 'norm')
TIMESERIES_TYPE = str(PARCELLATION.get("timeseries_type", "")).strip().lower()
if TIMESERIES_TYPE not in ("mean", "norm"):
    raise ValueError(
        f"Invalid config['parcellation']['timeseries_type'] = '{TIMESERIES_TYPE}'. "
        "Must be 'mean' or 'norm'.")

EPS_ZERO_SERIES = float(config.get("zero_threshold", 0.0))

# -------------------------
# Atlas selection + validation (Craddock | Schaefer | MIST)
# -------------------------
atlases_block = config.get("atlases", {})

# Canonical template bookkeeping (Craddock block used as source of MNI label/res defaults)
cr_cfg = atlases_block.get("Craddock", {})
CANONICAL_MNI    = cr_cfg.get("canonical_mni", "MNI152NLin2009cAsym")
CANONICAL_RES_MM = int(cr_cfg.get("canonical_res", 2))

# Atlas-family-specific validation + derived tag
if AF_LOWER == "craddock":
    CRADDOCK_DIR = Path(cr_cfg.get("craddock_dir", "")).expanduser()
    if not CRADDOCK_DIR.exists():
        raise FileNotFoundError(f"Craddock atlas directory not found: {CRADDOCK_DIR}")

    allowed_cr_counts = [int(x) for x in cr_cfg.get("allowed_num_ROIs", [50, 100, 200])]
    if ATLAS_N_ROIS not in allowed_cr_counts:
        raise ValueError(
            f"Craddock: requested n_rois={ATLAS_N_ROIS}, but allowed_num_ROIs={allowed_cr_counts}")

    # Craddock ignores network_scale
    if ATLAS_NETWORK_SCALE not in (None, "", "None"):
        print("[INFO] Craddock atlas does not use 'network_scale'; ignoring provided value.")
    ATLAS_NETWORK_SCALE = None

    ATLAS_TAG = f"craddock-{ATLAS_N_ROIS:03d}"

elif AF_LOWER == "schaefer":
    sch_cfg = atlases_block.get("Schaefer", {})

    # Schaefer requires network_scale and validates (n_rois, network_scale) combination
    if ATLAS_NETWORK_SCALE in (None, "", "None"):
        raise ValueError("Schaefer atlas requires 'parcellation.network_scale' (e.g., 7 or 17).")
    ATLAS_NETWORK_SCALE = int(ATLAS_NETWORK_SCALE)

    allowed_scales = sch_cfg.get("allowed_scales", {})  # e.g., {100:[7,17], 200:[7,17], ...}
    if allowed_scales and (ATLAS_N_ROIS not in allowed_scales):
        raise ValueError(
            f"Schaefer: requested n_rois={ATLAS_N_ROIS}, but allowed_scales keys are {list(allowed_scales.keys())}")
    if allowed_scales:
        allowed_networks = [int(x) for x in allowed_scales[ATLAS_N_ROIS]]
        if allowed_networks and (ATLAS_NETWORK_SCALE not in allowed_networks):
            raise ValueError(
                f"Schaefer: requested n_rois={ATLAS_N_ROIS} with network_scale={ATLAS_NETWORK_SCALE}, "
                f"but allowed_scales[{ATLAS_N_ROIS}] = {allowed_networks}")

    # If Schaefer block defines a preferred MNI resolution, honor it; else fall back to Craddock canonical_res
    CANONICAL_RES_MM = int(sch_cfg.get("resolution", CANONICAL_RES_MM))

    ATLAS_TAG = f"schaefer-{ATLAS_N_ROIS:03d}p-{ATLAS_NETWORK_SCALE}net"

elif AF_LOWER == "mist":
    mist_cfg = atlases_block.get("MIST", {})

    allowed_rois = [int(x) for x in mist_cfg.get("allowed_num_ROIs", [])]
    if allowed_rois and (ATLAS_N_ROIS not in allowed_rois):
        raise ValueError(
            f"MIST: requested n_rois={ATLAS_N_ROIS}, but allowed_num_ROIs={allowed_rois}")

    # MIST ignores network_scale
    if ATLAS_NETWORK_SCALE not in (None, "", "None"):
        print("[INFO] MIST atlas ignores 'network_scale'; using n_rois only.")
    ATLAS_NETWORK_SCALE = None

    ATLAS_TAG = f"mist-{ATLAS_N_ROIS}"

else:
    raise ValueError(
        f"Unsupported atlas family: {ATLAS_FAMILY!r} (expected 'Craddock', 'Schaefer', or 'MIST')")

print(
    f"\n[ATLAS] family={ATLAS_FAMILY} | n_rois={ATLAS_N_ROIS} | "
    f"network_scale={ATLAS_NETWORK_SCALE} | tag='{ATLAS_TAG}' | "
    f"canonical={CANONICAL_MNI}@{CANONICAL_RES_MM}mm")

# --------------------------------------------------------------------
### SET PATHS:

ROOT_DIR = Path(config["root_output_directory"]).expanduser()

# INPUTS:
RUN_MANIFEST_PATH   = ROOT_DIR / "subject_manifest.csv"
MEG_PARAMETERS_PATH = ROOT_DIR / "MEG_manifest.csv"

# This is the only required *data* input at this stage: fsaverage-morphed 4D volumes
MORPHS_DIRECTORY = ROOT_DIR / str(config["morph_output_dir"])

# OUTPUTS (prefix-based schema will be used later when writing):
PARCELLATION_OUTPUT_DIR = ROOT_DIR / str(config["parcellation_output_dir"])
PARCELLATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# QC output folder is flat (no subdirs):
QC_OUTPUT_DIR = None
if ENABLE_QC_OVERLAYS:
    QC_OUTPUT_DIR = ROOT_DIR / str(config["QC_output_dir"])
    QC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# (Optional) Atlas cache root used later for MNI→fsaverage atlas caching
ATLAS_CACHE_DIR = PARCELLATION_OUTPUT_DIR / "_atlas_cache"
ATLAS_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------------
### INITIALIZATION:

# Load DataFrames:
RUN_MANIFEST = pd.read_csv(RUN_MANIFEST_PATH)
MEG_runs     = pd.read_csv(MEG_PARAMETERS_PATH)

In [ ]:
# DIAGNOSTIC SUBSETTING (if enabled):
if isinstance(SUBSET, int) and SUBSET > 0:
    print(f"\nDIAGNOSTIC SUBSETTING ENABLED: Running only {SUBSET} test rows from MEG_manifest.csv")
    MEG_runs = MEG_runs.head(SUBSET).copy()
    display(MEG_runs)

------------------

In [ ]:
# =====================================================================
# CELL: ATLAS ACQUISITION (Craddock/Schaefer/MIST) + MNI HARMONIZATION
#        + CACHE ATLAS IN FSAVERAGE SPACE (via ANTs MNI<-->fsaverage T1/brain)
#        FIXES: use fsaverage brain.mgz for registration + brainmask clipping
#        UPDATE: OVERWRITE_CACHED_ATLAS support (recompute-by-default)
# =====================================================================

# -------------------------
# Helper utilities
# -------------------------

def _is_integer_like(arr, atol=1e-6):
    a = np.asanyarray(arr)
    a = a[np.isfinite(a)]
    if a.size == 0:
        return True
    return float(np.max(np.abs(a - np.rint(a)))) <= float(atol)

def _coerce_labels_integer(img):
    dat = img.get_fdata(dtype=np.float32)
    if not _is_integer_like(dat):
        print("[WARN] Atlas labels not strictly integer-like; rounding to nearest int.")
        dat = np.rint(dat)
    dat = dat.astype(np.int32, copy=False)
    dat[dat < 0] = 0
    return nib.Nifti1Image(dat, img.affine, img.header)

def _nonzero_stats_nib(img):
    d = np.asanyarray(img.get_fdata())
    nz = int((d != 0).sum())
    return nz, float(np.nanmin(d)) if d.size else 0.0, float(np.nanmax(d)) if d.size else 0.0

def _write_int16_nifti_like(src_img, data, out_path: Path):
    data_i16 = np.asanyarray(data).astype(np.int16, copy=False)
    out = nib.Nifti1Image(data_i16, src_img.affine, src_img.header)
    nib.save(out, str(out_path))

def _safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p

def _affine_close(a, b, tol=1e-4):
    return np.allclose(a, b, atol=tol, rtol=0)

def _pick_fsaverage_mgz(subjects_dir: Path) -> Tuple[Path, str]:
    """
    Prefer brain-extracted fsaverage brain.mgz. Fall back to T1.mgz.
    Returns (path, tag) where tag is 'brain' or 't1'.
    """
    cand_brain = subjects_dir / "fsaverage" / "mri" / "brain.mgz"
    cand_t1    = subjects_dir / "fsaverage" / "mri" / "T1.mgz"
    if cand_brain.exists():
        return cand_brain, "brain"
    if cand_t1.exists():
        return cand_t1, "t1"
    raise FileNotFoundError(
        f"Could not find fsaverage brain.mgz or T1.mgz under {subjects_dir / 'fsaverage' / 'mri'}")

# -------------------------
# Cache paths
# -------------------------

# Harmonized atlas + MNI ref
ATLAS_MNI_PATH = ATLAS_CACHE_DIR / f"atlas_{ATLAS_TAG}_mni{CANONICAL_RES_MM}mm.nii.gz"
MNI_REF_PATH   = ATLAS_CACHE_DIR / f"mni_ref_{CANONICAL_MNI}_{CANONICAL_RES_MM}mm.nii.gz"

# Ensure SUBJECTS_DIR is set
subj_dir_env = os.environ.get("SUBJECTS_DIR", "")
if not subj_dir_env:
    raise RuntimeError(
        "SUBJECTS_DIR environment variable is not set. "
        "Set it before running (must contain 'fsaverage').")
SUBJECTS_DIR_PATH = Path(subj_dir_env)
if not SUBJECTS_DIR_PATH.exists():
    raise RuntimeError(f"SUBJECTS_DIR does not exist: {SUBJECTS_DIR_PATH}")

# Pick fsaverage moving image for ANTs
FSAVERAGE_MGZ, FSAVERAGE_MGZ_TAG = _pick_fsaverage_mgz(SUBJECTS_DIR_PATH)

# fsaverage NIfTI for ANTs (cache depends on which mgz we used)
FSAVERAGE_NII = ATLAS_CACHE_DIR / f"fsaverage_{FSAVERAGE_MGZ_TAG}_for_ants.nii.gz"

# fsaverage --> MNI transform cache (depends on moving image tag + REG_TYPE + MNI res)
XFM_DIR = _safe_mkdir(
    ATLAS_CACHE_DIR / f"xfm_fsaverage_{FSAVERAGE_MGZ_TAG}_to_mni_{CANONICAL_RES_MM}mm_{str(REG_TYPE)}")
FS2MNI_AFF   = XFM_DIR / "0GenericAffine.mat"
FS2MNI_WARP  = XFM_DIR / "1Warp.nii.gz"
FS2MNI_INVW  = XFM_DIR / "1InverseWarp.nii.gz"
FS2MNI_PROV  = XFM_DIR / "fsaverage_to_mni_provenance.json"

# Final cached atlas on fsaverage space (depends on moving image tag)
ATLAS_FSAVG_PATH = ATLAS_CACHE_DIR / f"atlas_{ATLAS_TAG}_on_fsaverage_{FSAVERAGE_MGZ_TAG}.nii.gz"

# For hard-clipping (brainmask)
FSAVERAGE_BRAINMASK_MGZ = SUBJECTS_DIR_PATH / "fsaverage" / "mri" / "brainmask.mgz"

# -------------------------
# Optional: hard overwrite -> purge cached artifacts first
# -------------------------
if OVERWRITE_CACHED_ATLAS:
    print("[CACHE] Overwrite enabled: removing existing cached atlas/xfm artifacts (if present).")
    for p in [
        ATLAS_MNI_PATH,
        ATLAS_FSAVG_PATH,
        MNI_REF_PATH,
        FSAVERAGE_NII,
        FS2MNI_AFF,
        FS2MNI_WARP,
        FS2MNI_INVW,
        FS2MNI_PROV]:
        try:
            if p.exists():
                p.unlink()
                print(f"        removed: {p.name}")
        except Exception as e:
            print(f"        [WARN] could not remove {p.name}: {e}")

# -------------------------
# Step 1: Resolve source atlas (MNI) & harmonize to canonical MNI grid
# -------------------------

print("\n==============================================================")
print("[STEP 1] Resolve atlas source (family-specific) + harmonize in MNI")
print("==============================================================")

# 1a) Resolve source atlas file path (family-specific)
if AF_LOWER == "craddock":
    brain_map_path = CRADDOCK_DIR / f"Craddock-{ATLAS_N_ROIS}_ROIs.nii"
    if not brain_map_path.exists():
        raise FileNotFoundError(f"Craddock atlas not found at: {brain_map_path}")
    family_desc = f"Craddock {ATLAS_N_ROIS}-ROI atlas (local)"

elif AF_LOWER == "schaefer":
    print(
        f"[ATLAS] Fetching Schaefer-2018 via nilearn: n_rois={ATLAS_N_ROIS}, "
        f"yeo_networks={ATLAS_NETWORK_SCALE}, res={CANONICAL_RES_MM}mm")
    sch = datasets.fetch_atlas_schaefer_2018(
        n_rois=int(ATLAS_N_ROIS),
        yeo_networks=int(ATLAS_NETWORK_SCALE),
        resolution_mm=int(CANONICAL_RES_MM))
    brain_map_path = Path(sch.maps)
    family_desc = f"Schaefer-2018 {ATLAS_N_ROIS}-parcel, {ATLAS_NETWORK_SCALE}-network atlas (nilearn)"

elif AF_LOWER == "mist":
    print(f"[ATLAS] Fetching MIST/BASC multiscale via nilearn: scale={ATLAS_N_ROIS}")
    mist = datasets.fetch_atlas_basc_multiscale_2015()
    key = f"scale{int(ATLAS_N_ROIS):03d}"
    if not hasattr(mist, key):
        avail = [k for k in mist.keys() if k.startswith("scale")]
        raise RuntimeError(f"MIST fetcher missing {key}. Available keys: {avail}")
    brain_map_path = Path(getattr(mist, key))
    family_desc = f"MIST/BASC multiscale atlas @ {ATLAS_N_ROIS} parcels (nilearn)"

else:
    raise RuntimeError(f"Unexpected atlas family: {ATLAS_FAMILY!r}")

print(f"[ATLAS] Source: {family_desc}")
print(f"[ATLAS] Path:   {brain_map_path}")

# 1b) Load and coerce labels integer
src_atlas_img = nib.load(str(brain_map_path))
src_atlas_int = _coerce_labels_integer(src_atlas_img)

print("[ATLAS] Original shape:", src_atlas_img.shape)
print("[ATLAS] Original zooms:", src_atlas_img.header.get_zooms()[:3])
src_nz, src_min, src_max = _nonzero_stats_nib(src_atlas_int)
print(f"[ATLAS] Original int-coerced: nonzero_vox={src_nz:,} range={src_min:.1f}/{src_max:.1f}")

# 1c) Build canonical MNI ref @ CANONICAL_RES_MM
mni_ref_img = load_mni152_template(resolution=int(CANONICAL_RES_MM))

# Also create a simple MNI mask for registration stability
mni_masker = NiftiMasker(standardize=False)
mni_masker.fit(mni_ref_img)
mni_mask_img = nib.Nifti1Image(
    mni_masker.mask_img_.get_fdata().astype(np.uint8),
    mni_ref_img.affine,
    mni_ref_img.header)

if OVERWRITE_CACHED_ATLAS or (not MNI_REF_PATH.exists()):
    nib.save(mni_ref_img, str(MNI_REF_PATH))
    print(f"[CACHE] wrote MNI ref: {MNI_REF_PATH.name}")
else:
    print(f"[CACHE] using existing MNI ref: {MNI_REF_PATH.name}")

# 1d) Harmonize atlas onto canonical MNI ref grid (nearest)
if OVERWRITE_CACHED_ATLAS or (not ATLAS_MNI_PATH.exists()):
    atlas_on_mni = resample_to_img(
        src_atlas_int,
        mni_ref_img,
        interpolation="nearest",
        fill_value=0)
    atlas_dat_i32 = np.rint(atlas_on_mni.get_fdata()).astype(np.int32, copy=False)
    atlas_dat_i32[atlas_dat_i32 < 0] = 0
    atlas_on_mni = nib.Nifti1Image(atlas_dat_i32, atlas_on_mni.affine, atlas_on_mni.header)

    _write_int16_nifti_like(atlas_on_mni, atlas_dat_i32, ATLAS_MNI_PATH)
    print(f"[CACHE] wrote harmonized atlas: {ATLAS_MNI_PATH.name}")
else:
    print(f"[CACHE] using existing harmonized atlas: {ATLAS_MNI_PATH.name}")

# Optional quick sanity plot on MNI template
try:
    plotting.plot_roi(
        str(ATLAS_MNI_PATH),
        bg_img=str(MNI_REF_PATH),
        title=f"{ATLAS_TAG} (harmonized) over {CANONICAL_MNI} {CANONICAL_RES_MM}mm",
        display_mode="ortho",
        draw_cross=False)
    plt.show()
except Exception as e:
    print(f"[WARN] MNI overlay QC plot failed: {e}")

# -------------------------
# Step 2: Compute / reuse fsaverage --> MNI transforms (ANTs)
# -------------------------

print("\n==============================================================")
print("[STEP 2] Compute/reuse fsaverage --> MNI transforms (ANTs)")
print("==============================================================")

print(f"[FSAVG] Moving image for registration: {FSAVERAGE_MGZ_TAG} ({FSAVERAGE_MGZ.name})")
print(f"[REG ] type_of_transform = {REG_TYPE}")

# Convert fsaverage mgz -> NIfTI once (reused for registration/apply)
if OVERWRITE_CACHED_ATLAS or (not FSAVERAGE_NII.exists()):
    img = nib.load(str(FSAVERAGE_MGZ))
    nii = nib.Nifti1Image(img.get_fdata(dtype=np.float32), img.affine)
    nib.save(nii, str(FSAVERAGE_NII))
    print(f"[CACHE] wrote fsaverage {FSAVERAGE_MGZ_TAG} NIfTI for ANTs: {FSAVERAGE_NII.name}")
else:
    print(f"[CACHE] using existing fsaverage {FSAVERAGE_MGZ_TAG} NIfTI: {FSAVERAGE_NII.name}")

need_xfm = (
    OVERWRITE_CACHED_ATLAS or
    not (FS2MNI_AFF.exists() and FS2MNI_WARP.exists() and FS2MNI_INVW.exists()))

if need_xfm:
    try:
        import ants
    except Exception as e:
        raise RuntimeError(f"ANTsPy is required to compute transforms: {e}")

    # Save temp MNI mask for ANTs
    tmp_mni_mask = ATLAS_CACHE_DIR / f"_tmp_mni_mask_{CANONICAL_RES_MM}mm.nii.gz"
    if OVERWRITE_CACHED_ATLAS or (not tmp_mni_mask.exists()):
        nib.save(mni_mask_img, str(tmp_mni_mask))

    ants_fixed  = ants.image_read(str(MNI_REF_PATH))      # fixed = MNI
    ants_fmask  = ants.image_read(str(tmp_mni_mask))      # mask in fixed space
    ants_moving = ants.image_read(str(FSAVERAGE_NII))     # moving = fsaverage brain/T1

    print(
        f"[REG] ants.registration fixed=MNI({ants_fixed.shape}) "
        f"moving=fsaverage_{FSAVERAGE_MGZ_TAG}({ants_moving.shape}) type={REG_TYPE}")

    reg = ants.registration(
        fixed=ants_fixed,
        moving=ants_moving,
        type_of_transform=str(REG_TYPE),
        mask=ants_fmask,
        verbose=False)

    fwd = list(reg["fwdtransforms"])
    inv = list(reg["invtransforms"])

    src_aff  = next(p for p in fwd if p.endswith(".mat"))
    src_warp = next(p for p in fwd if p.endswith(".gz"))
    src_invw = next(p for p in inv if p.endswith(".gz"))

    shutil.copy2(src_aff,  str(FS2MNI_AFF))
    shutil.copy2(src_warp, str(FS2MNI_WARP))
    shutil.copy2(src_invw, str(FS2MNI_INVW))

    prov = {
        "transform_set": f"fsaverage_{FSAVERAGE_MGZ_TAG}_to_MNI (ANTs)",
        "type_of_transform": str(REG_TYPE),
        "fixed": {"space": f"{CANONICAL_MNI}_{CANONICAL_RES_MM}mm", "ref": str(MNI_REF_PATH)},
        "moving": {"space": f"fsaverage_{FSAVERAGE_MGZ_TAG}", "ref": str(FSAVERAGE_NII)},
        "outputs": {"affine": str(FS2MNI_AFF), "warp": str(FS2MNI_WARP), "invwarp": str(FS2MNI_INVW)}}
    with open(FS2MNI_PROV, "w") as f:
        json.dump(prov, f, indent=2)

    print(f"[CACHE] wrote transforms: {FS2MNI_AFF.name}, {FS2MNI_WARP.name}, {FS2MNI_INVW.name}")
else:
    print("[CACHE] using existing fsaverage --> MNI transform set")

# -------------------------
# Step 3: Apply MNI --> fsaverage to atlas + hard-clip to brainmask, cache atlas_on_fsaverage
# -------------------------

print("\n==============================================================")
print("[STEP 3] Apply MNI --> fsaverage to atlas (nearest), clip to brainmask, cache atlas_on_fsaverage")
print("==============================================================")

if OVERWRITE_CACHED_ATLAS or (not ATLAS_FSAVG_PATH.exists()):
    try:
        import ants
    except Exception as e:
        raise RuntimeError(f"ANTsPy is required to apply transforms: {e}")

    ants_fixed = ants.image_read(str(FSAVERAGE_NII))   # fixed target (fsaverage brain/T1 grid)
    ants_atlas = ants.image_read(str(ATLAS_MNI_PATH))  # moving atlas in MNI

    def _nonzero_stats_ants(ants_img):
        arr = ants_img.numpy()
        nz  = int((arr != 0).sum())
        return nz, float(arr.min(initial=0.0)), float(arr.max(initial=0.0))

    # Two plausible inverse orderings (mirrors fMRI safety check)
    A = ants.apply_transforms(
        fixed=ants_fixed,
        moving=ants_atlas,
        transformlist=[str(FS2MNI_INVW), str(FS2MNI_AFF)],
        whichtoinvert=[False, True],
        interpolator="nearestNeighbor")
    nzA, _, _ = _nonzero_stats_ants(A)

    B = ants.apply_transforms(
        fixed=ants_fixed,
        moving=ants_atlas,
        transformlist=[str(FS2MNI_AFF), str(FS2MNI_INVW)],
        whichtoinvert=[True, False],
        interpolator="nearestNeighbor")
    nzB, _, _ = _nonzero_stats_ants(B)

    chosen = "A" if nzA >= nzB else "B"
    atlas_fs = A if chosen == "A" else B
    nzF, fmin, fmax = _nonzero_stats_ants(atlas_fs)

    print(
        f"[VARIANT] atlas MNI-->fsaverage_{FSAVERAGE_MGZ_TAG}: nzA={nzA:,} nzB={nzB:,} -> chosen={chosen} "
        f"| nz_final={nzF:,} range={fmin:.1f}/{fmax:.1f}")

    if nzF == 0:
        raise RuntimeError("MNI --> fsaverage atlas projection produced an empty label map (all zeros).")

    # Write atlas as int16 (preserve image info)
    atlas_arr = atlas_fs.numpy().astype(np.int16, copy=False)
    out = ants.from_numpy(atlas_arr)
    out = ants.copy_image_info(atlas_fs, out)
    ants.image_write(out, str(ATLAS_FSAVG_PATH))

    # ---- Hard clip to fsaverage brainmask (recommended) ----
    if FSAVERAGE_BRAINMASK_MGZ.exists():
        atl_img = nib.load(str(ATLAS_FSAVG_PATH))
        bm = nib.load(str(FSAVERAGE_BRAINMASK_MGZ))

        # Resample mask -> atlas grid (nearest)
        bm_rs = resample_from_to(
            bm,
            (atl_img.shape[:3], atl_img.affine), order=0)
        bm_rs_dat = (bm_rs.get_fdata() > 0)

        atl_dat = np.rint(atl_img.get_fdata()).astype(np.int16, copy=False)
        before_nz = int((atl_dat != 0).sum())
        atl_dat[~bm_rs_dat] = 0
        after_nz = int((atl_dat != 0).sum())

        nib.save(nib.Nifti1Image(atl_dat, atl_img.affine, atl_img.header), str(ATLAS_FSAVG_PATH))
        print(f"[CLIP] Applied fsaverage brainmask: nz {before_nz:,} --> {after_nz:,}")
    else:
        print("[WARN] fsaverage brainmask.mgz not found; skipping hard clip.")

    # Report labels
    atlas_fs_nib = nib.load(str(ATLAS_FSAVG_PATH))
    labs = np.unique(np.rint(atlas_fs_nib.get_fdata()).astype(np.int32))
    n_labs = int((labs > 0).sum())
    print(
        f"[CACHE] wrote atlas_on_fsaverage: {ATLAS_FSAVG_PATH.name} | "
        f"unique_labels(incl 0)={labs.size} | nonzero_labels={n_labs}")

else:
    print(f"[CACHE] using existing atlas_on_fsaverage: {ATLAS_FSAVG_PATH.name}")

# Optional QC overlay on fsaverage (canonicalized for interpretable axes)
try:
    bg = nib.as_closest_canonical(nib.load(str(FSAVERAGE_NII)))
    roi = nib.as_closest_canonical(nib.load(str(ATLAS_FSAVG_PATH)))

    plotting.plot_roi(
        roi,
        bg_img=bg,
        title=f"{ATLAS_TAG} atlas on fsaverage_{FSAVERAGE_MGZ_TAG} (cached, canonicalized)",
        display_mode="ortho",
        draw_cross=False)
    plt.show()
except Exception as e:
    print(f"[WARN] fsaverage overlay QC plot failed: {e}")

In [ ]:
# ==============================================================
# CELL — FILETARGETS: Build + audit filetargets_df (MEG morphs)
#   Inputs live at:
#     MORPHS_DIRECTORY/<subject_ID>/<subject_ID>_<MEG_session_ID>-aligned.nii
#   Notes:
#     - MEG_runs uses MEG_session_ID (not session_ID)
#     - prefix is constructed as "<subject_ID>_<MEG_session_ID>"
# ==============================================================

print("\n==============================================================")
print("[FILETARGETS] Build + audit filetargets_df (MEG morphs)")
print("==============================================================")

# -------------------------
# 0) Basic validation
# -------------------------

if "MEG_runs" not in globals() or not isinstance(MEG_runs, pd.DataFrame):
    raise RuntimeError("MEG_runs DataFrame not found. Run the init cell first.")

required_meg_cols = ["subject_ID", "MEG_session_ID"]
for c in required_meg_cols:
    if c not in MEG_runs.columns:
        raise RuntimeError(f"MEG_runs missing required column: {c}")

MORPHS_DIRECTORY = Path(MORPHS_DIRECTORY)
if not MORPHS_DIRECTORY.exists():
    raise RuntimeError(f"MORPHS_DIRECTORY does not exist: {MORPHS_DIRECTORY}")

# Atlas cache should exist from the previous atlas cell
if "ATLAS_FSAVG_PATH" not in globals():
    raise RuntimeError("ATLAS_FSAVG_PATH not found. Run the atlas acquisition/harmonization cell first.")

atlas_exists = Path(ATLAS_FSAVG_PATH).exists()
print(f"[INFO] MORPHS_DIRECTORY = {MORPHS_DIRECTORY}")
print(f"[INFO] Atlas(fsaverage) exists? {atlas_exists} | {ATLAS_FSAVG_PATH}")

# Optional diagnostic subsetting (respect the already-subsetted MEG_runs)
df = MEG_runs.copy()
print(f"[INFO] MEG_runs rows to process: {len(df)}")
print("[INFO] Using prefix = '<subject_ID>_<MEG_session_ID>'")

# -------------------------
# 1) Resolver for morph NIfTI
# -------------------------

def _resolve_morph_nii(subject_id, meg_session_id, morphs_dir):
    """
    Expected:
      morphs_dir/<subject_id>/<subject_id>_<meg_session_id>-aligned.nii[.gz]
    Returns (path_or_None, note)
    """
    subject_id = str(subject_id).strip()
    meg_session_id = str(meg_session_id).strip()

    subj_dir = Path(morphs_dir) / subject_id
    if not subj_dir.exists():
        return None, "missing_subject_dir"

    # Prefer strict exact match first
    candidates = []
    exact_patterns = [
        f"{subject_id}_{meg_session_id}-aligned.nii",
        f"{subject_id}_{meg_session_id}-aligned.nii.gz",
        f"{subject_id}_{meg_session_id}_aligned.nii",
        f"{subject_id}_{meg_session_id}_aligned.nii.gz"]
    for pat in exact_patterns:
        p = subj_dir / pat
        if p.exists():
            candidates.append(p)

    if candidates:
        # If multiple exact matches (nii and nii.gz), prefer .nii.gz
        candidates_sorted = sorted(candidates, key=lambda p: (p.name.endswith(".nii.gz"), p.stat().st_mtime), reverse=True)
        return candidates_sorted[0], "exact_match"

    # Fallback: scan for any aligned nifti containing both subject and session token
    hits = []
    for p in subj_dir.glob("*.nii*"):
        name = p.name
        if "aligned" not in name.lower():
            continue
        if subject_id not in name:
            continue
        if meg_session_id not in name:
            continue
        hits.append(p)

    if not hits:
        return None, "no_matching_aligned_nii_in_subject_dir"

    # Prefer newest if multiple
    hits_sorted = sorted(hits, key=lambda p: p.stat().st_mtime, reverse=True)
    return hits_sorted[0], "fuzzy_match"

# -------------------------
# 2) Build filetargets_df
# -------------------------

rows = []
for _, r in df.iterrows():
    subject_id = str(r["subject_ID"]).strip()
    meg_sess   = str(r["MEG_session_ID"]).strip()
    prefix     = f"{subject_id}_{meg_sess}"

    nii_path, note = _resolve_morph_nii(subject_id, meg_sess, MORPHS_DIRECTORY)

    rows.append({
        "subject_ID": subject_id,
        "MEG_session_ID": meg_sess,
        "prefix": prefix,
        "meg_fsaverage_input": str(nii_path) if nii_path else "missing",
        "meg_input_note": note,
        "atlas_fsaverage": str(ATLAS_FSAVG_PATH) if atlas_exists else "missing",
        "missing_meg_fsaverage_input": (nii_path is None),
        "missing_atlas_fsaverage": (not atlas_exists)})

filetargets_df = pd.DataFrame(rows)

# -------------------------
# 3) Audit summary
# -------------------------

rows_total = int(len(filetargets_df))
rows_missing_meg = int(filetargets_df["missing_meg_fsaverage_input"].sum())
rows_missing_atlas = int(filetargets_df["missing_atlas_fsaverage"].sum())
rows_ok = int(((~filetargets_df["missing_meg_fsaverage_input"]) & (~filetargets_df["missing_atlas_fsaverage"])).sum())

print("\n[AUDIT] Summary")
print(f"  rows_total                       = {rows_total}")
print(f"  rows_ok                          = {rows_ok}")
print(f"  rows_missing_meg_fsaverage_input = {rows_missing_meg}")
print(f"  rows_missing_atlas_fsaverage     = {rows_missing_atlas}")

# Show a compact view of problems (if any)
if rows_missing_meg > 0:
    print("\n[AUDIT] Missing MEG inputs (first 15):")
    miss = filetargets_df[filetargets_df["missing_meg_fsaverage_input"]].head(15)
    for _, rr in miss.iterrows():
        print(f"  - {rr['prefix']} | subject_dir={Path(MORPHS_DIRECTORY)/rr['subject_ID']} | note={rr['meg_input_note']}")

# Show first few resolved paths for sanity
print("\n[AUDIT] Example resolved inputs (first 10):")
ex = filetargets_df[filetargets_df["meg_fsaverage_input"] != "missing"].head(10)
for _, rr in ex.iterrows():
    print(f"  - {rr['prefix']} -> {rr['meg_fsaverage_input']} ({rr['meg_input_note']})")

In [ ]:
# =====================================================================
# CELL — PARCELLATE MEG MORPHS (fsaverage) -> ROI TIMESERIES (MEAN or NORM)
#        Output schema matches fMRI branch (except t_ columns are 5-digit)
# =====================================================================

print("\n==============================================================")
print("[PARCELLATE] Extract ROI timeseries from MEG morphs (fsaverage)")
print("==============================================================")

# -------------------------
# Required globals (sanity)
# -------------------------
timeseries_type = str(globals().get("TIMESERIES_TYPE", "")).strip().lower()
if timeseries_type not in ("mean", "norm"):
    raise ValueError(
        "Invalid TIMESERIES_TYPE from config. Must be 'mean' or 'norm'. "
        f"Got: {timeseries_type!r}")

OVERWRITE = bool(globals().get("OVERWRITE_PARCELLATIONS", False))
HARD_STOP = bool(globals().get("HARD_STOP", True))
EPS_ZERO_SERIES = float(globals().get("EPS_ZERO_SERIES", 0.0))

atlas_path = Path(globals().get("ATLAS_FSAVG_PATH", ""))
if (not atlas_path) or (not atlas_path.exists()):
    raise RuntimeError(f"ATLAS_FSAVG_PATH not found: {atlas_path}")

out_root = Path(globals().get("PARCELLATION_OUTPUT_DIR", ""))
if (not out_root):
    raise RuntimeError("PARCELLATION_OUTPUT_DIR is not defined.")
out_root.mkdir(parents=True, exist_ok=True)

if "filetargets_df" not in globals() or not isinstance(filetargets_df, pd.DataFrame):
    raise RuntimeError("filetargets_df is missing (run the MEG filetargets build cell first).")

# Choose which column contains the MEG morph NIfTI path:
candidate_cols = ["meg_fsaverage_input", "meg_morph_path", "meg_fsaverage_path", "morph_path", "input_path", "meg_path"]
morph_col = next((c for c in candidate_cols if c in filetargets_df.columns), None)
if morph_col is None:
    raise RuntimeError(
        "Could not find MEG morph path column in filetargets_df. "
        f"Tried: {candidate_cols}. Available columns: {list(filetargets_df.columns)}")

required_cols = ["prefix", morph_col]
for c in required_cols:
    if c not in filetargets_df.columns:
        raise RuntimeError(f"filetargets_df missing required column: {c}")

# -------------------------
# Helper utilities (match fMRI)
# -------------------------
def _affine_close(a, b, tol=1e-4):
    return np.allclose(a, b, atol=tol, rtol=0)

def _zscore_sample(X, axis=0):
    X = np.asarray(X, dtype=np.float32)
    with np.errstate(invalid="ignore", divide="ignore"):
        mu = np.nanmean(X, axis=axis, keepdims=True)
        sd = np.nanstd(X, axis=axis, ddof=1, keepdims=True)
        bad = (~np.isfinite(sd)) | (sd == 0)
        sd[bad] = 1.0
        Z = (X - mu) / sd
        Z[~np.isfinite(Z)] = 0.0
    return Z

def _is_integer_like(arr, atol=1e-6):
    a = np.asanyarray(arr)
    a = a[np.isfinite(a)]
    if a.size == 0:
        return True
    return np.max(np.abs(a - np.rint(a))) <= atol

def _load_atlas_on_target_grid(atlas_nifti_path, target_img):
    """
    Ensure atlas is on the same (X,Y,Z) grid + affine as target_img.
    If mismatch, resample atlas to target using nearest neighbor.
    Returns int32 atlas labels array on target grid.
    """
    atlas_img = nib.load(str(atlas_nifti_path))
    atlas_dat = atlas_img.get_fdata(dtype=np.float32)

    if not _is_integer_like(atlas_dat):
        atlas_dat = np.rint(atlas_dat)

    atlas_lab = atlas_dat.astype(np.int32, copy=False)
    atlas_lab[atlas_lab < 0] = 0

    same_shape = (atlas_img.shape[:3] == target_img.shape[:3])
    same_aff   = _affine_close(atlas_img.affine, target_img.affine)

    if same_shape and same_aff:
        return atlas_lab, atlas_img, False

    # Resample to target lattice
    atlas_rs = resample_from_to(
        nib.Nifti1Image(atlas_lab.astype(np.int16, copy=False), atlas_img.affine, atlas_img.header),
        (target_img.shape[:3], target_img.affine),
        order=0)
    atlas_rs_dat = np.rint(atlas_rs.get_fdata(dtype=np.float32)).astype(np.int32, copy=False)
    atlas_rs_dat[atlas_rs_dat < 0] = 0
    return atlas_rs_dat, atlas_rs, True

# -------------------------
# Execution
# -------------------------
n_total = len(filetargets_df)
wrote = 0
skipped_existing = 0
failed = 0

for r in filetargets_df.itertuples(index=False):
    prefix = str(getattr(r, "prefix"))
    in_path = Path(getattr(r, morph_col))

    if not in_path.exists():
        print(f"[SKIP] {prefix}: missing input morph: {in_path}")
        failed += 1
        continue

    # Output directory + filenames (match fMRI naming stem)
    out_dir = out_root / prefix
    out_dir.mkdir(parents=True, exist_ok=True)

    ts_token     = timeseries_type.upper()
    atlas_token  = str(globals().get("ATLAS_FAMILY", "Atlas"))
    n_rois_token = f"{int(globals().get('ATLAS_N_ROIS', 0)):03d}"

    name_parts = [prefix, ts_token, atlas_token, n_rois_token]
    if str(globals().get("AF_LOWER", "")).lower() == "schaefer":
        net = globals().get("ATLAS_NETWORK_SCALE", None)
        if net is not None and str(net).strip() not in ("", "None"):
            name_parts.append("net" + str(int(net)))

    base_stem = "_".join(name_parts)
    csv_out = out_dir / f"{base_stem}.csv"
    json_sidecar = out_dir / f"{base_stem}_info.json"

    if (not OVERWRITE) and csv_out.exists() and json_sidecar.exists():
        skipped_existing += 1
        continue

    # Load MEG morphed volume (expect 4D: X,Y,Z,T)
    img = nib.load(str(in_path))
    dat = img.get_fdata(dtype=np.float32)
    if dat.ndim != 4:
        print(f"[SKIP] {prefix}: input is not 4D (got shape={dat.shape}).")
        failed += 1
        continue

    X, Y, Z, T = dat.shape

    # Load atlas and ensure lattice match (resample if needed)
    atlas_lab, atlas_img_used, atlas_was_resampled = _load_atlas_on_target_grid(atlas_path, img)

    # ROI labels
    labels = np.unique(atlas_lab[atlas_lab > 0])
    if labels.size == 0:
        print(f"[SKIP] {prefix}: atlas has no labels > 0 after grid match.")
        failed += 1
        continue

    if atlas_was_resampled:
        print(f"[INFO] {prefix}: atlas resampled to MEG grid (nearest).")

    print(f"[EXTRACT] {prefix}: T={T} | ROIs={labels.size} | atlas_tag={globals().get('ATLAS_TAG','')}")
    # Flatten for efficient indexing
    data_2d = dat.reshape(-1, T)         # (V x T)
    labs_flat = atlas_lab.reshape(-1)    # (V,)

    # Per-ROI voxel indices and counts
    roi_indices = {}
    roi_counts = {}
    for lab in labels:
        lab_int = int(lab)
        idx = np.where(labs_flat == lab_int)[0]
        roi_indices[lab_int] = idx
        roi_counts[lab_int] = int(idx.size)

    # Extract mean per ROI (T x N)
    N = labels.size
    mean_ts = np.zeros((T, N), dtype=np.float32)
    for j, lab in enumerate(labels):
        lab_int = int(lab)
        idx = roi_indices[lab_int]
        if idx.size == 0:
            continue
        mean_ts[:, j] = np.nanmean(data_2d[idx, :], axis=0, dtype=np.float64)

    # Z-score (sample z per ROI) – identical to fMRI
    z_ts = _zscore_sample(mean_ts, axis=0)

    # Diagnostics: empty-by-count & effectively-zero
    empty_by_count = [int(lab) for lab in labels if roi_counts[int(lab)] == 0]
    absmax = np.max(np.abs(mean_ts), axis=0)
    zero_like_mask = absmax < EPS_ZERO_SERIES
    n_zero_like = int(np.sum(zero_like_mask))
    zero_like_labels = [int(labels[i]) for i in np.where(zero_like_mask)[0]]

    if n_zero_like > 0:
        weakest_idx = np.argsort(absmax)[: min(5, N)]
        weak_report = ", ".join(
            f"{int(labels[i])} (max|x|={absmax[i]:.2e}, vox={roi_counts[int(labels[i])]})"
            for i in weakest_idx)
        print(f"    Weakest ROIs: {weak_report}")

    labels_sorted = sorted(labels.tolist())
    labels_contiguous = (
        labels_sorted == list(range(int(labels_sorted[0]), int(labels_sorted[0]) + N)))
    labels_from_one = (labels_sorted == list(range(1, N + 1)))

    mean_naninf = (not np.isfinite(mean_ts).all())
    norm_naninf = (not np.isfinite(z_ts).all())
    mean_std = np.std(mean_ts, axis=0, ddof=1)
    norm_std = np.std(z_ts, axis=0, ddof=1)
    mean_const = int((mean_std < 1e-12).sum())
    norm_const = int((norm_std < 1e-12).sum())
    norm_mu_abs = float(np.mean(np.abs(np.mean(z_ts, axis=0))))
    norm_std_avg = float(np.mean(norm_std))

    print(
        f"  --> [REPORT]  {prefix}: total ROIs={N} | empty-by-count={len(empty_by_count)} "
        f"| effectively-zero={n_zero_like}")
    print(
        f"  --> [REPORT+] {prefix}: labels_contiguous={labels_contiguous}"
        f"{' (1..N)' if labels_from_one else ''} | "
        f"MEAN: NaN/Inf={mean_naninf}, const_cols={mean_const} | "
        f"NORM: NaN/Inf={norm_naninf}, const_cols={norm_const}, "
        f"avg|mu|={norm_mu_abs:.4f}, avgstd={norm_std_avg:.4f}")

    # Build row-wise (ROI) DataFrames with 5-digit time columns for MEG
    t_cols = [f"t_{i:05d}" for i in range(1, T + 1)]
    roi_ids = [int(x) for x in labels.tolist()]
    vox_cts = [int(roi_counts[int(l)]) for l in labels]

    mean_df = (
        pd.DataFrame(np.transpose(mean_ts), index=roi_ids, columns=t_cols)
          .reset_index()
          .rename(columns={"index": "ROI"}))
    mean_df.insert(1, "num_voxels", vox_cts)

    norm_df = (
        pd.DataFrame(np.transpose(z_ts), index=roi_ids, columns=t_cols)
          .reset_index()
          .rename(columns={"index": "ROI"}))
    norm_df.insert(1, "num_voxels", vox_cts)

    if timeseries_type == "mean":
        out_df = mean_df
    elif timeseries_type == "norm":
        out_df = norm_df
    else:
        raise RuntimeError(f"Unexpected TIMESERIES_TYPE at save-time: {timeseries_type!r}")

    out_df.to_csv(csv_out, index=False, float_format="%.10g")

    # JSON sidecar: mirror fMRI schema as closely as possible
    meta = {
        "prefix": prefix,
        "n_timepoints": int(T),
        "n_rois": int(N),
        "atlas": {
            "family": str(globals().get("ATLAS_FAMILY", "")),
            "tag": str(globals().get("ATLAS_TAG", "")),
            "n_rois_requested": int(globals().get("ATLAS_N_ROIS", N)),
            "network_scale": int(globals().get("ATLAS_NETWORK_SCALE", 0)) if globals().get("ATLAS_NETWORK_SCALE", None) is not None else None,
            # Keep key for cross-branch compatibility; MEG does not use EPI filenames
            "epi_filename": None,
            "fsaverage_filename": str(atlas_path.name),
            "atlas_was_resampled_to_meg_grid": bool(atlas_was_resampled)},
        "processing": {
            "apply_gm_clip": False,
            "eps_zero_series": float(EPS_ZERO_SERIES),
            "timeseries_type": timeseries_type},
        "time_axis": {
            "indexing": "1-based post-trim",
            "columns_pattern": f"t_00001..t_{T:05d}"},
        "roi_labels": [int(x) for x in labels.tolist()],
        "roi_voxel_counts_after_mask": {str(k): int(v) for k, v in roi_counts.items()},
        "n_empty_rois_by_count": int(len(empty_by_count)),
        "n_effectively_zero_series": int(n_zero_like),
        "effectively_zero_labels": [int(x) for x in zero_like_labels],
        "csv_diagnostics": {
            "labels_contiguous": bool(labels_contiguous),
            "labels_from_one": bool(labels_from_one),
            "mean_nan_or_inf": bool(mean_naninf),
            "norm_nan_or_inf": bool(norm_naninf),
            "mean_const_cols": int(mean_const),
            "norm_const_cols": int(norm_const),
            "norm_avg_abs_mu": float(norm_mu_abs),
            "norm_avg_std": float(norm_std_avg)},
        "inputs": {
            "meg_morph_path": str(in_path),
            "atlas_path": str(atlas_path)},
        "outputs": {
            "timeseries_type": timeseries_type,
            "csv": str(csv_out),
            "json_sidecar": str(json_sidecar)}}

    with open(json_sidecar, "w") as f:
        json.dump(meta, f, indent=2)

    wrote += 1
    print(f"[OK] {prefix}: {timeseries_type.upper()} -> {csv_out.name} | atlas_tag={globals().get('ATLAS_TAG','')}")

print("\n[SUMMARY]")
print(f"  rows_total       = {n_total}")
print(f"  wrote            = {wrote}")
print(f"  skipped_existing = {skipped_existing}")
print(f"  failed           = {failed}")

if HARD_STOP and failed > 0:
    raise RuntimeError(
        f"{failed} rows failed during extraction. "
        "Set hard_errors: False to allow partial completion.")

In [ ]:
# =====================================================================
# QC CELL (NO PLOTS): Overlap metrics + ROI voxel-count QC + timeseries QC
#   - Keeps overlap table (support vs atlas)
#   - Adds:
#       A) ROI voxel counts per subject (zero-ROI flags)
#       B) Timeseries distribution sanity (mean_ts)
#       C) Z-score behavior checks (norm_ts) when TIMESERIES_TYPE='norm'
#       D) Atlas resampling consistency checks across subjects
# =====================================================================

print("\n==============================================================")
print("[QC] Overlap metrics + ROI voxel-count + timeseries QC (NO PLOTS)")
print("==============================================================")

# -------------------------
# Required globals (sanity)
# -------------------------
if "filetargets_df" not in globals() or not isinstance(filetargets_df, pd.DataFrame):
    raise RuntimeError("filetargets_df not found. Run filetargets build cell first.")

atlas_path = Path(globals().get("ATLAS_FSAVG_PATH", ""))
if (not atlas_path) or (not atlas_path.exists()):
    raise RuntimeError(f"ATLAS_FSAVG_PATH not found: {atlas_path}")

out_root = Path(globals().get("PARCELLATION_OUTPUT_DIR", ""))
if not out_root:
    raise RuntimeError("PARCELLATION_OUTPUT_DIR is not defined.")
out_root.mkdir(parents=True, exist_ok=True)

ENABLE_QC_OVERLAYS = bool(globals().get("ENABLE_QC_OVERLAYS", True))
QC_OUTPUT_DIR = globals().get("QC_OUTPUT_DIR", None)
if QC_OUTPUT_DIR is None:
    # Fall back to writing QC into parcellation output root if QC_OUTPUT_DIR not defined
    QC_OUTPUT_DIR = out_root / "_QC"
QC_OUTPUT_DIR = Path(QC_OUTPUT_DIR)
QC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"[QC] Writing outputs to: {QC_OUTPUT_DIR}")

timeseries_type = str(globals().get("TIMESERIES_TYPE", "")).strip().lower()
if timeseries_type not in ("mean", "norm"):
    raise RuntimeError(f"TIMESERIES_TYPE must be 'mean' or 'norm'. Got: {timeseries_type!r}")

EPS_ZERO_SERIES = float(globals().get("EPS_ZERO_SERIES", 0.0))

# Identify morph column in filetargets_df
candidate_cols = ["meg_fsaverage_input", "meg_morph_path", "meg_fsaverage_path", "morph_path", "input_path", "meg_path"]
morph_col = next((c for c in candidate_cols if c in filetargets_df.columns), None)
if morph_col is None:
    raise RuntimeError(
        "Could not find MEG morph path column in filetargets_df. "
        f"Tried: {candidate_cols}. Available: {list(filetargets_df.columns)}")
print(f"[QC] Using morph_col='{morph_col}'")

# -------------------------
# Helper utilities
# -------------------------
def _affine_close(a, b, tol=1e-4):
    return np.allclose(a, b, atol=tol, rtol=0)

def _is_integer_like(arr, atol=1e-6):
    a = np.asanyarray(arr)
    a = a[np.isfinite(a)]
    if a.size == 0:
        return True
    return float(np.max(np.abs(a - np.rint(a)))) <= float(atol)

def _load_atlas_on_target_grid(atlas_nifti_path, target_img):
    """
    Ensure atlas is on the same (X,Y,Z) grid + affine as target_img.
    If mismatch, resample atlas to target using nearest neighbor.
    Returns:
      atlas_lab (int32 on target grid), atlas_was_resampled (bool),
      shape_match (bool), affine_match (bool)
    """
    from nibabel.processing import resample_from_to

    atlas_img = nib.load(str(atlas_nifti_path))
    atlas_dat = atlas_img.get_fdata(dtype=np.float32)

    if not _is_integer_like(atlas_dat):
        atlas_dat = np.rint(atlas_dat)

    atlas_lab = atlas_dat.astype(np.int32, copy=False)
    atlas_lab[atlas_lab < 0] = 0

    shape_match = (atlas_img.shape[:3] == target_img.shape[:3])
    affine_match = _affine_close(atlas_img.affine, target_img.affine)

    if shape_match and affine_match:
        return atlas_lab, False, True, True

    atlas_rs = resample_from_to(
        nib.Nifti1Image(atlas_lab.astype(np.int16, copy=False), atlas_img.affine, atlas_img.header),
        (target_img.shape[:3], target_img.affine), order=0)
    atlas_rs_dat = np.rint(atlas_rs.get_fdata(dtype=np.float32)).astype(np.int32, copy=False)
    atlas_rs_dat[atlas_rs_dat < 0] = 0
    return atlas_rs_dat, True, shape_match, affine_match

def _dice(a_bool, b_bool):
    a = a_bool.reshape(-1)
    b = b_bool.reshape(-1)
    inter = int(np.logical_and(a, b).sum())
    a_sum = int(a.sum())
    b_sum = int(b.sum())
    denom = (a_sum + b_sum)
    if denom == 0:
        return 0.0
    return 2.0 * inter / float(denom)

def _safe_zscore_sample(X, axis=0):
    # Must match fMRI branch behavior
    X = np.asarray(X, dtype=np.float32)
    with np.errstate(invalid="ignore", divide="ignore"):
        mu = np.nanmean(X, axis=axis, keepdims=True)
        sd = np.nanstd(X, axis=axis, ddof=1, keepdims=True)
        bad = (~np.isfinite(sd)) | (sd == 0)
        sd[bad] = 1.0
        Z = (X - mu) / sd
        Z[~np.isfinite(Z)] = 0.0
    return Z

# -------------------------
# QC parameters for overlap support mask
# -------------------------
support_percentile = float(globals().get("QC_SUPPORT_PERCENTILE", 98.0))
min_support_voxels = int(globals().get("QC_MIN_SUPPORT_VOXELS", 200))

print(f"[QC] Support percentile: {support_percentile} | min_support_voxels: {min_support_voxels}")

# -------------------------
# Main loop: compute overlap + ROI voxel-count QC + timeseries QC
# -------------------------
rows_total = int(len(filetargets_df))
ok = 0
skipped = 0

overlap_records = []
roi_count_records = []
ts_records = []
atlas_signature_records = []

for r in filetargets_df.itertuples(index=False):
    prefix = str(getattr(r, "prefix"))
    in_path = Path(getattr(r, morph_col))

    if (not in_path.exists()):
        skipped += 1
        overlap_records.append({
            "prefix": prefix,
            "status": "missing_input",
            "input_path": str(in_path)})
        continue

    img = nib.load(str(in_path))
    dat = img.get_fdata(dtype=np.float32)
    if dat.ndim != 4:
        skipped += 1
        overlap_records.append({
            "prefix": prefix,
            "status": "not_4d",
            "input_path": str(in_path),
            "shape": str(getattr(dat, "shape", None))})
        continue

    X, Y, Z, T = dat.shape

    # Atlas on target grid (same helper used in extraction)
    atlas_lab, atlas_was_resampled, shape_match_atlas, affine_match_atlas = _load_atlas_on_target_grid(atlas_path, img)
    atlas_mask = (atlas_lab > 0)

    # -------------------------
    # Atlas resampling consistency signature
    # -------------------------
    labels = np.unique(atlas_lab[atlas_lab > 0]).astype(np.int32)
    labels_sorted = np.sort(labels)
    n_labels = int(labels_sorted.size)

    # Voxel count per label (signature)
    # Note: for speed, compute counts via bincount on flattened labels
    labs_flat = atlas_lab.reshape(-1)
    max_lab = int(labels_sorted.max()) if n_labels > 0 else 0
    bc = np.bincount(labs_flat, minlength=max_lab + 1) if max_lab > 0 else np.array([0], dtype=np.int64)
    counts_vec = bc[labels_sorted] if n_labels > 0 else np.array([], dtype=np.int64)

    # Small stable "signature" for comparing across subjects
    # (sum, number of labels, min/max count, and a simple hash-like checksum)
    checksum = int(np.sum((counts_vec.astype(np.int64) * (np.arange(n_labels, dtype=np.int64) + 1)) % 1000003)) if n_labels > 0 else 0
    atlas_signature_records.append({
        "prefix": prefix,
        "atlas_was_resampled": bool(atlas_was_resampled),
        "shape_match_atlas": bool(shape_match_atlas),
        "affine_match_atlas": bool(affine_match_atlas),
        "n_labels_gt0": n_labels,
        "atlas_vox_gt0": int(atlas_mask.sum()),
        "label_min": int(labels_sorted.min()) if n_labels > 0 else None,
        "label_max": int(labels_sorted.max()) if n_labels > 0 else None,
        "count_min": int(counts_vec.min()) if n_labels > 0 else None,
        "count_max": int(counts_vec.max()) if n_labels > 0 else None,
        "counts_checksum": checksum,
        "input_path": str(in_path)})

    # -------------------------
    # A) ROI voxel counts per subject (+ zero-roi flags)
    # -------------------------
    zero_rois = []
    if n_labels > 0:
        zero_rois = [int(lab) for lab, ct in zip(labels_sorted.tolist(), counts_vec.tolist()) if int(ct) == 0]

    roi_count_records.append({
        "prefix": prefix,
        "n_rois": int(n_labels),
        "n_zero_rois": int(len(zero_rois)),
        "zero_rois": ",".join(str(x) for x in zero_rois[:50]) + ("..." if len(zero_rois) > 50 else ""),
        "atlas_vox_gt0": int(atlas_mask.sum()),
        "input_path": str(in_path)})

    # -------------------------
    # Overlap metrics (support vs atlas)
    # -------------------------
    # Support definition: top percentile of absolute signal across all voxels/time
    abs_dat = np.abs(dat)
    # Collapse time using max (robust and cheap) -> per-voxel score
    voxel_score = np.max(abs_dat, axis=3)  # (X,Y,Z)

    thr = float(np.percentile(voxel_score.reshape(-1), support_percentile))
    support_mask = (voxel_score >= thr)

    # Ensure minimum support voxels: if too small, relax threshold to meet min_support_voxels
    support_vox = int(support_mask.sum())
    if support_vox < min_support_voxels:
        flat = voxel_score.reshape(-1)
        # pick kth largest to get at least min_support_voxels
        k = min_support_voxels
        if k < flat.size:
            kth = np.partition(flat, flat.size - k)[flat.size - k]
            thr = float(kth)
            support_mask = (voxel_score >= thr)
            support_vox = int(support_mask.sum())

    atlas_vox = int(atlas_mask.sum())
    inter_vox = int(np.logical_and(support_mask, atlas_mask).sum())

    support_inside_atlas_frac = float(inter_vox) / float(support_vox) if support_vox > 0 else 0.0
    atlas_covered_by_support_frac = float(inter_vox) / float(atlas_vox) if atlas_vox > 0 else 0.0
    dice = float(_dice(support_mask, atlas_mask))

    overlap_records.append({
        "prefix": prefix,
        "status": "ok",
        "support_inside_atlas_frac": support_inside_atlas_frac,
        "atlas_covered_by_support_frac": atlas_covered_by_support_frac,
        "dice": dice,
        "support_vox": support_vox,
        "atlas_vox": atlas_vox,
        "support_thr": thr,
        "support_percentile": support_percentile,
        "shape_match_atlas": bool(shape_match_atlas),
        "affine_match_atlas": bool(affine_match_atlas),
        "atlas_was_resampled": bool(atlas_was_resampled),
        "input_path": str(in_path)})

    # -------------------------
    # Timeseries distribution checks
    #   - Recompute mean_ts quickly (same logic as extraction, but without writing)
    # -------------------------
    data_2d = dat.reshape(-1, T)            # (V,T)
    labs_flat = atlas_lab.reshape(-1)       # (V,)

    N = n_labels
    mean_ts = np.zeros((T, N), dtype=np.float32)
    for j, lab in enumerate(labels_sorted):
        idx = np.where(labs_flat == int(lab))[0]
        if idx.size == 0:
            continue
        mean_ts[:, j] = np.nanmean(data_2d[idx, :], axis=0, dtype=np.float64)

    # B) mean_ts basic stats
    mean_ts_finite = np.isfinite(mean_ts)
    mean_absmax = float(np.max(np.abs(mean_ts[mean_ts_finite]))) if mean_ts_finite.any() else 0.0
    mean_naninf = bool((not np.isfinite(mean_ts).all()))
    mean_std = np.std(mean_ts, axis=0, ddof=1)
    mean_const = int((mean_std < 1e-12).sum())

    # effectively-zero columns by EPS threshold (same as extraction logic)
    absmax_by_roi = np.max(np.abs(mean_ts), axis=0)
    zero_like_mask = absmax_by_roi < EPS_ZERO_SERIES
    n_zero_like = int(np.sum(zero_like_mask))

    rec = {
        "prefix": prefix,
        "T": int(T),
        "N_rois": int(N),
        "mean_nan_or_inf": mean_naninf,
        "mean_absmax": mean_absmax,
        "mean_const_cols": mean_const,
        "n_effectively_zero_series": n_zero_like,
        "eps_zero_series": float(EPS_ZERO_SERIES)}

    # C) z-score checks (only meaningful if norm is in use, but we can still compute diagnostics)
    z_ts = _safe_zscore_sample(mean_ts, axis=0)
    norm_naninf = bool((not np.isfinite(z_ts).all()))
    norm_std = np.std(z_ts, axis=0, ddof=1)
    norm_const = int((norm_std < 1e-12).sum())
    norm_mu_abs = float(np.mean(np.abs(np.mean(z_ts, axis=0))))
    norm_std_avg = float(np.mean(norm_std))

    rec.update({
        "norm_nan_or_inf": norm_naninf,
        "norm_const_cols": norm_const,
        "norm_avg_abs_mu": norm_mu_abs,
        "norm_avg_std": norm_std_avg})

    ts_records.append(rec)

    ok += 1

# -------------------------
# Write outputs
# -------------------------
overlap_df = pd.DataFrame(overlap_records)
roi_counts_df = pd.DataFrame(roi_count_records)
ts_df = pd.DataFrame(ts_records)
atlas_sig_df = pd.DataFrame(atlas_signature_records)

overlap_csv = QC_OUTPUT_DIR / "QC_support_overlap_metrics.csv"
roi_counts_csv = QC_OUTPUT_DIR / "QC_roi_voxel_counts_summary.csv"
ts_csv = QC_OUTPUT_DIR / "QC_timeseries_distribution_summary.csv"
atlas_sig_csv = QC_OUTPUT_DIR / "QC_atlas_resampling_signature.csv"

overlap_df.to_csv(overlap_csv, index=False)
roi_counts_df.to_csv(roi_counts_csv, index=False)
ts_df.to_csv(ts_csv, index=False)
atlas_sig_df.to_csv(atlas_sig_csv, index=False)

print("\n[QC SUMMARY]")
print(f"  rows_total = {rows_total}")
print(f"  ok         = {ok}")
print(f"  skipped    = {skipped}")
print(f"  overlap_csv   = {overlap_csv}")
print(f"  roi_counts_csv= {roi_counts_csv}")
print(f"  ts_csv        = {ts_csv}")
print(f"  atlas_sig_csv = {atlas_sig_csv}")

# -------------------------
# Console views: key triage tables
# -------------------------
# 1) Overlap: show lowest support_inside_atlas_frac
if "support_inside_atlas_frac" in overlap_df.columns:
    print("\n[QC] Lowest support_inside_atlas_frac (first 10)")
    cols = [
        "prefix", "support_inside_atlas_frac", "atlas_covered_by_support_frac", "dice",
        "support_vox", "atlas_vox", "support_thr", "shape_match_atlas", "affine_match_atlas",
        "atlas_was_resampled", "input_path"]
    cols = [c for c in cols if c in overlap_df.columns]
    display(overlap_df[overlap_df["status"] == "ok"].sort_values("support_inside_atlas_frac").head(10)[cols])

# 2) ROI counts: show subjects with most zero ROIs
if len(roi_counts_df) > 0:
    print("\n[QC] Subjects with most zero-voxel ROIs (first 10)")
    cols = ["prefix", "n_rois", "n_zero_rois", "atlas_vox_gt0", "zero_rois", "input_path"]
    cols = [c for c in cols if c in roi_counts_df.columns]
    display(roi_counts_df.sort_values(["n_zero_rois", "atlas_vox_gt0"], ascending=[False, True]).head(10)[cols])

# 3) Timeseries distribution: quick triage
if len(ts_df) > 0:
    print("\n[QC] Timeseries distribution triage (first 10 worst by norm_avg_abs_mu, then norm_avg_std)")
    cols = [
        "prefix", "T", "N_rois",
        "mean_nan_or_inf", "mean_const_cols", "n_effectively_zero_series", "mean_absmax",
        "norm_nan_or_inf", "norm_const_cols", "norm_avg_abs_mu", "norm_avg_std"]
    cols = [c for c in cols if c in ts_df.columns]
    display(ts_df.sort_values(["norm_avg_abs_mu", "norm_avg_std"], ascending=[False, True]).head(10)[cols])

# 4) Atlas signature consistency: how many distinct signatures?
if len(atlas_sig_df) > 0:
    n_unique_checksums = int(atlas_sig_df["counts_checksum"].nunique())
    n_unique_labelsets = int(atlas_sig_df[["n_labels_gt0", "label_min", "label_max"]].drop_duplicates().shape[0])
    print("\n[QC] Atlas resampling consistency summary")
    print(f"  unique count checksums: {n_unique_checksums}")
    print(f"  unique (n_labels,label_min,label_max): {n_unique_labelsets}")

    # Show any outliers in labelset
    base = atlas_sig_df[["n_labels_gt0", "label_min", "label_max"]].mode().iloc[0].to_dict()
    out = atlas_sig_df[
        (atlas_sig_df["n_labels_gt0"] != base["n_labels_gt0"]) |
        (atlas_sig_df["label_min"] != base["label_min"]) |
        (atlas_sig_df["label_max"] != base["label_max"])]
    if len(out) > 0:
        print("\n[QC] WARNING: label-set outliers detected (first 10)")
        display(out.head(10)[["prefix", "n_labels_gt0", "label_min", "label_max", "atlas_vox_gt0", "input_path"]])
    else:
        print("  label-set appears consistent across subjects (no outliers detected).")

----------